# Cyber Security Attack Analysis
**Course:** Data Visualization — Team 04

This notebook runs the full workflow specified in the project proposal:
1. **Data Profiling, Cleaning & Validation** (`Datasets/cybersecurity_attacks.csv` → `Datasets/cleaned_cybersecurity_attacks.csv`)
2. **Interactive Plotly Visualizations** (10 charts using the colorblind-safe Okabe-Ito palette & ordered light-to-dark severity scale)
3. **Static Exports** (10 high-DPI PNGs + combined multi-page PDF in `exports/`)
4. **Interactive KPI Dashboard** (`dashboard.html` with Attack Type, Severity, and Year filters)
5. **Key Findings Verification** (Section 6 summary statistics)

In [1]:
import pandas as pd
import plotly.express as px

import analysis

print("Project directory:", analysis.BASE)

Project directory: D:\DV\Data-Visualization Project


## 1. Data Profiling, Cleaning & Validation
- Recodes the 5 missing-value columns into `Yes`/`No` detection flags without imputation.
- Parses `Timestamp`, `Geo-location Data` (`City`, `State`), and `Device Information` (`Device OS`, `Browser Family`).
- Re-checks duplicates after type conversions, validates numeric ranges, adds derived fields (`High Anomaly`, `Anomaly Band`, `Packet Size Band`), and drops privacy-sensitive fields.

In [2]:
clean = analysis.clean_dataset()
clean.head()

Shape: (40000, 25)


Duplicate rows (raw): 0
Columns with missing values:
 Malware Indicators    20000
Alerts/Warnings       20067
Proxy Information     19851
Firewall Logs         19961
IDS/IPS Alerts        20050
dtype: int64 

IoC Detected: Yes=20,000  No=20,000
Alert Triggered: Yes=19,933  No=20,067
Proxy Used: Yes=20,149  No=19,851
Firewall Log: Yes=20,039  No=19,961
IDS/IPS Alert: Yes=19,950  No=20,050



Duplicate rows (post-conversion): 0
Ports out of range: 0
Packet length out of 64-1500: 0
Anomaly score out of 0-100: 0
High Anomaly (>=75) count: 10012



Saved Datasets\cleaned_cybersecurity_attacks.csv (40,000 rows, 37 cols)


,Timestamp,Source Port,Destination Port,Protocol,Packet Length,Packet Type,Traffic Type,Malware Indicators,Anomaly Scores,Alerts/Warnings,...,Browser Family,Year,Quarter,Month,DayOfWeek,Hour,Blocked,Anomaly Band,High Anomaly,Packet Size Band
0,2023-05-30 06:33:58,31225,17616,ICMP,503,Data,HTTP,IoC Detected,28.67,NaN,...,Internet Explorer,2023,2023Q2,2023-05-01,Tuesday,6,0,25-50,No,501-1000
1,2020-08-26 07:08:30,17245,48166,ICMP,1174,Data,HTTP,IoC Detected,51.50,NaN,...,Internet Explorer,2020,2020Q3,2020-08-01,Wednesday,7,1,50-75,No,1001-1500
2,2022-11-13 08:23:25,16811,53600,UDP,306,Control,HTTP,IoC Detected,87.42,Alert Triggered,...,Internet Explorer,2022,2022Q4,2022-11-01,Sunday,8,0,75-100,Yes,64-500
3,2023-07-02 10:38:46,20018,32534,UDP,385,Data,HTTP,NaN,15.79,Alert Triggered,...,Firefox,2023,2023Q3,2023-07-01,Sunday,10,1,0-25,No,64-500
4,2023-07-16 13:11:07,6131,26646,TCP,1462,Data,DNS,NaN,0.52,Alert Triggered,...,Internet Explorer,2023,2023Q3,2023-07-01,Sunday,13,1,0-25,No,1001-1500


## 2. Generate Interactive Charts, Static PNG/PDF Exports & Dashboard
Writes:
- `charts/*.html` (10 interactive Plotly charts)
- `exports/*.png` & `exports/cybersecurity_attack_charts.pdf`
- `dashboard.html` (interactive KPI row + Attack Type, Severity, and Year filters)

In [3]:
analysis.build_plotly_charts(clean)
analysis.build_static_exports(clean)
analysis.build_interactive_dashboard(clean)

Saved 11 PNGs and exports\cybersecurity_attack_charts.pdf in exports/


Saved interactive dashboard: dashboard.html


## 3. Inline Preview — Key Finding: Response Action by Severity

In [4]:
act = pd.crosstab(clean["Severity Level"], clean["Action Taken"], normalize="index").mul(100).round(1)
act_long = act.reset_index().melt(id_vars="Severity Level", var_name="Action Taken", value_name="Percent")
fig = px.bar(
    act_long,
    x="Severity Level",
    y="Percent",
    color="Action Taken",
    barmode="group",
    category_orders={"Severity Level": analysis.SEV_ORDER, "Action Taken": analysis.ACTION_ORDER},
    color_discrete_map=analysis.ACTION_COLORS,
    text="Percent",
    labels={"Percent": "Share of Events (%)"},
    title="Key Finding: Blocking rate (~34%) does not increase for High-severity attacks",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_yaxes(range=[0, 44], ticksuffix="%")
analysis.style_plotly(fig, show_legend=True).show()

## 4. Section 6 Key Findings Summary

In [5]:
analysis.print_summary(clean)


--- Numbers to compare with the proposal's Key Findings ---
Events per year:
 Year
2020    10573
2021    10538
2022    10750
2023     8139
Name: count, dtype: int64

Attack type %:
 Attack Type
DDoS         33.6
Malware      33.3
Intrusion    33.2
Name: proportion, dtype: float64

Severity %:
 Severity Level
Medium    33.6
High      33.5
Low       33.0
Name: proportion, dtype: float64

Action taken %:
 Action Taken
Blocked    33.8
Ignored    33.2
Logged     33.0
Name: proportion, dtype: float64

Action % within each severity:
 Action Taken    Blocked  Ignored  Logged
Severity Level                          
Low                34.0     33.4    32.6
Medium             33.6     32.8    33.6
High               33.9     33.3    32.8

Device OS counts:
 Device OS
Windows    17953
Linux       7171
macOS       5813
iOS         5774
Android     3289
Name: count, dtype: int64

Browser Family counts:
 Browser Family
Chrome               8105
Opera                8049
Internet Explorer    7995
Sa